In [7]:
COMMAND = (
    'python agent_interact.py '
    '--docker_name {docker_name} '
    '--container_name {container_name} '
    '--model_name {model_name} '
    '--task_dir {task_dir} '
    '--config_file {config_file} '
    '--tag {tag} '
    '--max_iter {max_iter} '
    '--mode {mode} '
    '--add_hints {add_hints}'
)

Single Run

In [23]:
TAG = 'test'

DOCKER_NAME = "officebench"
# MODEL_NAME = 'ToolQA/0123_universal-adhesive'
#MODEL_NAME = 'gpt-3.5-turbo-0125'
MODEL_NAME = 'llama3.1-70b'
CONTAINER_NAME = f'officebench-{TAG}'
# TASK_DIR = 'tasks/2-1'
TASK_DIR = 'tasks_train/1-7'
CONFIG_FILE = f'{TASK_DIR}/subtasks/4.json'
MODE = 'force_new'
MAX_ITER = 40
ADD_HINTS = True

print(COMMAND.format(
    docker_name=DOCKER_NAME,
    container_name=CONTAINER_NAME,
    model_name=MODEL_NAME,
    task_dir=TASK_DIR,
    config_file=CONFIG_FILE,
    tag=TAG,
    max_iter=MAX_ITER,
    mode=MODE,
    add_hints=ADD_HINTS
))

python agent_interact.py --docker_name officebench --container_name officebench-test --model_name llama3.1-70b --task_dir tasks_train/1-7 --config_file tasks_train/1-7/subtasks/4.json --tag test --max_iter 40 --mode force_new --add_hints True


Multiple Runs

In [ ]:
import glob

TAG = 'test'
SCRIPT_FILE = f'{TAG}.sh'

DOCKER_NAME = "officebench"

# MODEL_NAME = 'gpt-4o-2024-05-13'
MODEL_NAME = 'llama3.1-70b'

MAX_ITER = 40 # default value used in the paper
TASK_DIR = sorted(glob.glob('tasks/*'))
MODE = 'force_new'
ADD_HINTS = True

with open(SCRIPT_FILE, 'w') as fw:
    to_print = []
    for task_dir in TASK_DIR:
        for config_file in sorted(glob.glob(f'{task_dir}/subtasks/*.json')):
            command = COMMAND.format(
                docker_name=DOCKER_NAME,
                container_name=f'officebench-{TAG}',
                model_name=MODEL_NAME,
                task_dir=task_dir,
                config_file=config_file,
                tag=TAG,
                max_iter=MAX_ITER,
                mode=MODE,
                add_hints=ADD_HINTS
            )
            to_print.append((f'echo "Start {config_file}"', command, config_file))

    def sort_func(pair):
        tag = pair[2]
        app_num = int(tag.split('/')[1].split('-')[0])
        task_id = int(tag.split('/')[1].split('-')[1])
        subtask_id = int(tag.split('/')[-1].split('.')[0])
        return app_num, task_id, subtask_id

    to_print = sorted(to_print, key=sort_func)
    for pair in to_print:
        fw.write(pair[0] + '\n')
        fw.write(pair[1] + '\n')
        fw.write('\n')